In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
from py4j.protocol import Py4JJavaError
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Instance SparkSession
spark = SparkSession.builder\
        .appName('Analyze and Visualization Bitcoin')\
        .master('local[*]')\
        .getOrCreate()

# Extract Stage

## Read inputs

In [ ]:
# Define dictionary for input_paths
paths_list = [
    'data/output/cryptos',
    'data/output/market_bitcoin',
    'data/output/hist_bitcoin_prices',
    'data/output/moving_average_bitcoin'
]

def read_paths(paths):
    try:
        # Read CSV files into PySpark DataFrames
        cryptos_df = spark.read.parquet(paths[0])
        market_bitcoin_df = spark.read.parquet(paths[1])
        historical_bitcoin_prices_df = spark.read.parquet(paths[2])
        moving_average_bitcoin_df = spark.read.parquet(paths[3])
    
        # Save DataFrames in a dictionary
        df_collection_dict = {
            'cryptos': cryptos_df,
            'market_bitcoin': market_bitcoin_df,
            'historical_bitcoin_prices': historical_bitcoin_prices_df,
            'moving_average_bitcoin': moving_average_bitcoin_df
        }
    
        return df_collection_dict

    except Py4JJavaError as err:
        if 'org.apache.hadoop.security.AccessControlException' in str(err.java_exception):
            print('Permission denied: User does not have access to the specified file or directory.')
        else:
            print('An unexpected error occurred:', str(err))

    except AnalysisException as err:
        if 'Permission denied' in str(err):
            print('Permission denied: User does not have access to the specified file or directory.')
        elif 'cannot resolve' in str(err):
            print('Column not found: The specified column does not exist in the table.')
        elif 'Table or view not found' in str(err):
            print('Table not found: The specified table or view does not exist.')
        else:
            print('An AnalysisException occurred:', str(err))

    except Exception as err:
        print('An unexpected error occurred:', str(err))

    return None

In [ ]:
# Instance read_paths
dataframes_dict = read_paths(paths=paths_list)

# Trannsform Stage

## Convert Spark DataFrame to Pandas

In [ ]:
# Change sdf to pdf
cryptos_pdf = dataframes_dict['cryptos'].toPandas()
market_bitcoin_pdf = dataframes_dict['market_bitcoin'].toPandas()
historical_bitcoin_prices_pdf = dataframes_dict['historical_bitcoin_prices'].toPandas()
moving_average_bitcoin_pdf = dataframes_dict['moving_average_bitcoin'].toPandas()

## Analyze and Visualize Stage

## Crypto Symbol Initial Count

In [ ]:
cryptos_pdf["first_letter"] = cryptos_pdf["symbol"].str[0]
plt.figure(figsize=(15, 6))
sns.countplot(y=cryptos_pdf["first_letter"], order=cryptos_pdf["first_letter"].value_counts().index, hue=cryptos_pdf["first_letter"], palette="coolwarm", legend=False)
plt.title("Distribución de criptomonedas por símbolo")
plt.xlabel("Cantidad")
plt.ylabel("Símbolo de la criptomoneda")
plt.grid()
plt.show()

## Bitcoin Price in Different Currencies

In [ ]:
plt.figure(figsize=(20, 6))
sns.barplot(data=market_bitcoin_pdf, x="currency", y="current_price", hue="currency", palette="coolwarm", legend=False)

plt.title("Precio de Bitcoin en Diferentes Monedas")
plt.xlabel("Moneda")
plt.ylabel("Precio")
plt.xticks(rotation=45)
plt.grid()
plt.show()

## Bitcoin Price Evolution in January 2025

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=historical_bitcoin_prices_pdf, x="date", y="prices", color="royalblue")
plt.title("Evolución del Precio de Bitcoin en Enero 2025")
plt.xlabel("Fecha")
plt.ylabel("Precio en USD")
plt.xticks(rotation=45)
plt.grid()
plt.show()

## Bitcoin Price Evolution and Moving Average (January-March 2025)

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=moving_average_bitcoin_pdf, x="date", y="prices", label="Precio", color="royalblue")
sns.lineplot(data=moving_average_bitcoin_pdf, x="date", y="5_day_moving_avg", label="Media Móvil 5D", color="orange")
plt.title("Evolución del Precio de Bitcoin y su Media Móvil (Enero-Marzo 2025)")
plt.xlabel("Fechas")
plt.ylabel("Precio en USD")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()